# Chapter 17 Companion Notebook
**Build Your First LLM — Chapter 17: Deployment Options**

Open this local companion in Jupyter, or choose **File → Upload notebook** in Colab. The cells display example files and terminal instructions. They need no credentials or helper file, make no requests, and do not deploy anything.

If you choose to run the examples on Modal, use your own local terminal. Remote functions can incur charges; merely running the notebook cells cannot start a GPU.

In [ ]:
# The cells below print source code; install nothing in the notebook runtime.
print('Optional local terminal setup (use a virtual environment with Python 3.11):')
print('python -m pip install modal fastapi pydantic requests')
print('modal setup  # Authenticate the CLI; this is not a web proxy token.')

## Section 17.1: Why Modal?

**What is Serverless?**

| Traditional Server | Serverless (Modal) |
|-------------------|-------------------|
| You rent a computer 24/7 | Code runs only when needed |
| You pay even when idle | You pay per second of compute |
| You manage the host | Modal manages the host; you still control app access |
| Fixed capacity | Automatic scaling |

**Analogy:** Traditional hosting is like owning a car. Serverless is like using a taxi—you pay only when you ride.

**Why Modal for LLMs?**
- GPU support built-in (T4 to H100)
- $30/month free credits (~37 hours L4 GPU time; ~50 hours on T4)
- Python-native (no Docker, no YAML)
- Scales to zero (no GPU compute charge once idle containers stop)

## Section 17.2: Hello Modal

The simplest Modal app. Save this as `hello.py` and run with `modal run hello.py`:

In [ ]:
# Save this as hello.py
hello_modal_code = '''
import modal

app = modal.App("hello-world")

@app.function()
def hello(name: str) -> str:
    return f"Hello, {name}!"

@app.local_entrypoint()
def main():
    result = hello.remote("World")
    print(result)
'''

print("Save this code as 'hello.py':")
print(hello_modal_code)
print("\nThen run: modal run hello.py")

**What just happened?**

1. `modal.App()` creates your application
2. `@app.function()` marks code to run in Modal's cloud
3. `hello.remote()` calls the function remotely
4. Modal spun up a container, ran your code, returned the result

## Section 17.3: Adding GPU Support

One parameter adds GPU support:

In [ ]:
# Save this as gpu_test.py
gpu_test_code = '''
import modal

app = modal.App("gpu-test")

# torch must be installed in the container image (next section covers images)
image = modal.Image.debian_slim(python_version="3.11").pip_install("torch")

@app.function(image=image, gpu="L4")  # That's it!
def check_gpu():
    import torch
    if torch.cuda.is_available():
        device = torch.cuda.get_device_name(0)
        return f"GPU available: {device}"
    return "No GPU found"

@app.local_entrypoint()
def main():
    print(check_gpu.remote())
'''

print("Save this code as 'gpu_test.py':")
print(gpu_test_code)
print("\nThen run: modal run gpu_test.py")
print("\nExpected output: GPU available: NVIDIA L4")

**One parameter.** No CUDA installation, no driver management, no Docker images. Modal handles the entire GPU stack.

## Section 17.4: Installing Dependencies

Your LLM needs libraries. Modal lets you define your container environment in Python:

In [ ]:
# Save this as with_deps.py
with_deps_code = '''
import modal

# Define the container image
image = modal.Image.debian_slim(python_version="3.11").pip_install(
    "torch",
    "transformers",
    "accelerate",
)

app = modal.App("with-deps")

@app.function(image=image, gpu="T4")
def generate_text():
    from transformers import pipeline
    
    generator = pipeline("text-generation", model="gpt2", device=0)
    result = generator("The meaning of life is", max_length=50)
    return result[0]["generated_text"]

@app.local_entrypoint()
def main():
    print(generate_text.remote())
'''

print("Save this code as 'with_deps.py':")
print(with_deps_code)
print("\nThen run: modal run with_deps.py")

The `image` parameter tells Modal what to install. First run takes longer (building the image), but subsequent runs reuse the cached image.

## Section 17.5: Web Endpoints with FastAPI

This CPU-only demo has two routes: `/health` returns a static status and `/hello/{name}` returns a greeting. Protect the whole ASGI app, including its docs, with Modal proxy auth. A URL identifies the service; it is not permission to use it.

Before trying either web demo, create a **proxy token** in Modal Settings → Proxy auth tokens (or run `modal workspace proxy-tokens create --name chapter17` in a private terminal). Store the `wk-…` ID and one-time `ws-…` secret in your password manager. Use a dedicated demo environment and, on RBAC-enabled workspaces, allow the token only for that environment. A token is not restricted to one URL just because you name it after the app. Keep it with trusted operators, not in a notebook, browser frontend, screenshot or repository.

In [ ]:
# Save this as simple_api.py
simple_api_code = '''
import modal
from fastapi import FastAPI

app = modal.App("my-api")
web_app = FastAPI()

# The web stack runs INSIDE the Modal container, not on your laptop,
# so the image must carry it (pins match the book's tested environment):
# fastapi + pydantic for the app, uvicorn for the ASGI server, httpx for
# TestClient-style checks inside the container.
image = modal.Image.debian_slim(python_version="3.11").pip_install(
    "fastapi==0.163.2",
    "uvicorn==0.54.0",
    "httpx==0.28.1",
    "pydantic==2.14.3",
)

@web_app.get("/health")
def health():
    return {"status": "healthy"}

@web_app.get("/hello/{name}")
def hello(name: str):
    return {"message": f"Hello, {name}!"}

@app.function(image=image)
@modal.asgi_app(requires_proxy_auth=True)
def serve():
    return web_app
'''

print("Save this code as 'simple_api.py':")
print(simple_api_code)
print("\nThen run: modal deploy simple_api.py")
print("\nYou'll get a URL like: https://your-workspace--my-api-serve.modal.run")

**Optional terminal check.** In a private Bash session, set `SIMPLE_URL` to the exact HTTPS origin Modal printed for **my-api**. Load the proxy credentials from your secret manager into `MODAL_KEY` and `MODAL_SECRET` in that session. Do not type secret values into shell history or enable shell tracing.

```bash
curl --max-time 30 --header @- "$SIMPLE_URL/health" <<EOF
Modal-Key: $MODAL_KEY
Modal-Secret: $MODAL_SECRET
EOF
# Expected: {"status":"healthy"}. The same two headers are needed for /hello/World.
modal app stop my-api
```

A plain browser visit (or a request missing the proxy pair) is not a successful health check: Modal denies it before the function runs.

## Section 17.6: The Complete LLM Service

The GPU service runs pretrained Qwen3.5-2B on an L4. As in the CPU demo, Modal checks the proxy pair **before** starting or waking a container. `/health` checks vLLM readiness without generating. `/chat` additionally checks a separate application bearer token **inside** FastAPI, before inference. A caller with a valid proxy pair can still wake a GPU, even if `/chat` later returns 401.

In [ ]:
# Save this as llm_service.py
# Print the service source only. The local vLLM server is not a second web endpoint.
llm_service_code = '''
import modal
from fastapi import FastAPI, HTTPException, Request, Depends
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials
from fastapi.responses import JSONResponse
from pydantic import BaseModel, field_validator
import os
from collections import defaultdict
import threading
import uuid
from datetime import datetime, timedelta
import subprocess
import time
import requests

MODEL = "Qwen/Qwen3.5-2B"
# Note: no trust_remote_code needed. Current models like Qwen3.5 are
# natively supported by vLLM; only enable it for repos whose custom
# code you have reviewed, since it executes code from the model repo.

image = modal.Image.debian_slim(python_version="3.11").pip_install(
    "fastapi",
    "vllm==0.34.0",  # Pin for reproducibility (current stable, checked 2026-11)
    "torch",
    "transformers",
    "requests",
)

app = modal.App("my-llm-service")
model_cache = modal.Volume.from_name("model-cache", create_if_missing=True)
web_app = FastAPI(title="My LLM API", version="1.0.0")

# Declaring the bearer scheme gives Swagger's "Authorize" button and marks
# protected routes in the docs; require_token below does the actual check.
bearer_scheme = HTTPBearer(auto_error=False)

# ===== Chapter 16 guardrails, plus auth and token budgets =====
MAX_PROMPT_CHARS = 4000  # cheap first filter; the real budget is in tokens
API_TOKEN = os.environ.get("API_TOKEN", "")  # set via Modal secret, see below
MAX_INPUT_TOKENS = 2048  # templated input budget (with the output cap, fits 8192)
MAX_OUTPUT_TOKENS = 512  # per-request generation cap, not a billing limit
RATE_LIMIT = 20
RATE_WINDOW = timedelta(minutes=1)
request_counts: dict[str, list[datetime]] = defaultdict(list)
_rate_lock = threading.Lock()

def require_token(credentials: HTTPAuthorizationCredentials | None = Depends(bearer_scheme)):
    """Bearer-token auth: a shared GPU endpoint must not be open to the world."""
    if not API_TOKEN or credentials is None or credentials.credentials != API_TOKEN:
        raise HTTPException(401, "Missing or invalid token")

class ChatRequest(BaseModel):
    message: str
    max_tokens: int = 256

    @field_validator("message")
    @classmethod
    def not_empty(cls, v):
        if not v.strip():
            raise ValueError("Message cannot be empty")
        if len(v) > MAX_PROMPT_CHARS:
            raise ValueError(f"Message too long (max {MAX_PROMPT_CHARS} chars)")
        return v

    @field_validator("max_tokens")
    @classmethod
    def cap_output(cls, v):
        if v < 1:
            raise ValueError("max_tokens must be positive")
        # Clamp instead of reject: clients asking for more get the cap, not an error
        return min(v, MAX_OUTPUT_TOKENS)

class ChatResponse(BaseModel):
    response: str
    latency_ms: float

def check_rate_limit(request: Request):
    client_ip = request.client.host
    now = datetime.now()
    # Check-and-update under one lock: these endpoints run in FastAPI's
    # threadpool, so without it two concurrent requests could both read
    # len == RATE_LIMIT - 1 and both slip through.
    with _rate_lock:
        recent = [t for t in request_counts[client_ip] if now - t < RATE_WINDOW]
        if len(recent) >= RATE_LIMIT:
            request_counts[client_ip] = recent
            raise HTTPException(429, "Too many requests. Please wait a minute.")
        recent.append(now)
        request_counts[client_ip] = recent

def log_request(request_id, input_tokens, output_tokens, latency_ms, status):
    # Allowlisted metadata only: never log prompts, responses, or raw
    # exception text. Modal captures stdout.
    import json
    print(json.dumps({"ts": datetime.now().isoformat(),
                      "request_id": request_id,
                      "input_tokens": input_tokens,
                      "output_tokens": output_tokens,
                      "latency_ms": round(latency_ms, 2),
                      "status": status}))

# ===== vLLM server management (one per container) =====
_server = None
_tokenizer = None

def start_vllm():
    """Start vllm serve once per container and wait until it is ready."""
    global _server, _tokenizer
    if _server is None:
        _server = subprocess.Popen([
            "vllm", "serve", MODEL,
            "--host", "127.0.0.1",            # only the authenticated ASGI app faces clients
            "--port", "8000",
            "--max-model-len", "8192",       # cap context: keep KV cache small on L4's 24GB
            "--gpu-memory-utilization", "0.90",
            "--dtype", "bfloat16",           # Qwen3.5 weights; supported on L4, not T4
        ])
        # Wait for the server's own cheap /health endpoint
        for _ in range(120):
            try:
                requests.get("http://localhost:8000/health", timeout=2).raise_for_status()
                break
            except Exception:
                time.sleep(2)
        else:
            raise RuntimeError("vLLM server did not become ready")
        from transformers import AutoTokenizer
        _tokenizer = AutoTokenizer.from_pretrained(MODEL)
    return _tokenizer

@web_app.get("/health")
def health():
    """Cheap probe: ask the local vLLM server, no generation."""
    try:
        requests.get("http://localhost:8000/health", timeout=2).raise_for_status()
        return {"status": "healthy", "model": "Qwen3.5-2B"}
    except Exception:
        # 503, not 200: monitoring should see a degraded service as down
        return JSONResponse(status_code=503,
                            content={"status": "degraded", "model": "Qwen3.5-2B"})

@web_app.post("/chat", response_model=ChatResponse)
def chat(request: ChatRequest, http_request: Request,
         _auth: None = Depends(require_token)):
    """Generate a response: template the prompt, let vLLM do the inference."""
    check_rate_limit(http_request)
    request_id = uuid.uuid4().hex[:12]
    start = time.time()
    try:
        tokenizer = start_vllm()
        # Instruct models expect their chat template, not raw text.
        # apply_chat_template wraps the message in the model's special tokens.
        messages = [{"role": "user", "content": request.message}]
        # Real token budget: count the templated input with the same
        # tokenizer vLLM uses. Characters are not tokens. With one
        # conversation, apply_chat_template returns a dict-like batch
        # whose "input_ids" is the flat token list; len() of the batch
        # itself would count keys (2), not tokens.
        input_ids = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True,
            return_dict=True)["input_ids"]
        if len(input_ids) > MAX_INPUT_TOKENS:
            raise HTTPException(
                422, f"Message too long ({len(input_ids)} tokens; "
                     f"max {MAX_INPUT_TOKENS} input tokens)")
        # Forward the SAME ids we just counted: /v1/completions accepts
        # a token-id list as "prompt", so vLLM cannot re-tokenize the
        # text into something different from what we budgeted.
        r = requests.post(
            "http://localhost:8000/v1/completions",
            json={"model": MODEL, "prompt": input_ids,
                  "max_tokens": request.max_tokens, "temperature": 0.7},
            timeout=120,
        )
        r.raise_for_status()
        body = r.json()
        text = body["choices"][0]["text"].strip()
        latency_ms = (time.time() - start) * 1000
        log_request(request_id, len(input_ids),
                    body.get("usage", {}).get("completion_tokens", 0),
                    latency_ms, 200)
        return ChatResponse(response=text, latency_ms=round(latency_ms, 2))
    except HTTPException as e:
        log_request(request_id, 0, 0, (time.time() - start) * 1000, e.status_code)
        raise
    except requests.HTTPError as e:
        # vLLM itself rejected the request (e.g. context overflow): pass
        # 4xx through with a sanitized message, never the backend's text.
        status = e.response.status_code if e.response is not None else 502
        if 400 <= status < 500:
            log_request(request_id, 0, 0, (time.time() - start) * 1000, status)
            raise HTTPException(status, "The model rejected this request. "
                                        "Try a shorter message.")
        log_request(request_id, 0, 0, (time.time() - start) * 1000, 502)
        raise HTTPException(502, "Inference backend error. Please try again.")
    except Exception:
        log_request(request_id, 0, 0, (time.time() - start) * 1000, 500)
        raise HTTPException(500, "Generation failed. Please try again.")

@app.function(
    image=image,
    gpu="L4",
    timeout=600,
    scaledown_window=300,  # Keep warm for 5 minutes
    volumes={"/root/.cache/huggingface": model_cache},  # model downloads once
    secrets=[modal.Secret.from_name("llm-api-token")],  # provides API_TOKEN
)
@modal.concurrent(max_inputs=10)  # Safe: the vLLM server batches concurrent requests
@modal.asgi_app(requires_proxy_auth=True)  # Modal checks admission before GPU startup
def serve():
    start_vllm()  # Load once when the container starts, reuse for every request
    return web_app

'''

print("Save this code as 'llm_service.py':")
print(llm_service_code)

In [ ]:
# Save the printed client locally as check_service.py. No requests run in this notebook.
client_code = '''
import os
import requests

URL = os.environ["LLM_URL"].rstrip("/")  # exact HTTPS origin printed by your own deploy
if not URL.startswith("https://"):
    raise ValueError("LLM_URL must use HTTPS")
proxy = {"Modal-Key": os.environ["MODAL_KEY"], "Modal-Secret": os.environ["MODAL_SECRET"]}
chat = {**proxy, "Authorization": "Bearer " + os.environ["API_TOKEN"]}

# Check the outer gate first. This request must not wake a GPU.
r = requests.get(f"{URL}/health", timeout=30, allow_redirects=False)
assert r.status_code == 401, f"Expected proxy denial, got {r.status_code}"

r = requests.get(f"{URL}/health", headers=proxy, timeout=300, allow_redirects=False)
assert r.status_code == 200 and r.json()["status"] == "healthy", f"Readiness: {r.status_code}"

def ask(headers, payload):
    return requests.post(f"{URL}/chat", headers=headers, json=payload,
                         timeout=150, allow_redirects=False)

r = ask(proxy, {"message": "hi"})  # proxy admission is not app authorization
assert r.status_code == 401, f"Expected app denial, got {r.status_code}"
r = ask(chat, {"message": " "})
assert r.status_code == 422, f"Expected validation error, got {r.status_code}"
r = ask(chat, {"message": "What is Python?", "max_tokens": 64})
assert r.status_code == 200 and r.json()["response"].strip(), f"Chat: {r.status_code}"
print("Proxy, readiness, app auth, validation and chat checks passed.")
'''
print(client_code)


### First-Run Checklist

For an **optional live run** in your terminal, not this notebook:

1. In the target Modal environment, create the `llm-api-token` secret in the dashboard with key `API_TOKEN` and a long, randomly generated value. Only the GPU service needs this Modal secret. The workspace proxy token from §17.5 is separate; do not use your CLI credentials for either layer.
2. Save the printed service and client. Set a billing alert first; then run `modal deploy llm_service.py` if you intend to incur GPU charges. Set `LLM_URL` to the exact HTTPS origin from **your own** deploy output, not a link supplied by someone else. Make `API_TOKEN`, `MODAL_KEY` and `MODAL_SECRET` available to the private client process through your secret manager. Do not paste secrets into source, output, a URL, or a shared shell command.
3. Run `python check_service.py`. Expect a proxy 401 with no headers, then a 200 on `/health` with the proxy pair. A 503 **from the app** means its local vLLM health probe failed; check `modal app logs my-llm-service` for startup failures if readiness does not recover. Poll sparingly: an admitted readiness probe can start or keep a GPU warm even though it does not generate tokens.
4. With the proxy pair but a missing or wrong app bearer, `/chat` should return an **app** 401. With both layers valid, a blank message should get 422 and a valid one should get a nonempty answer. The 600-second Modal function timeout is not a client timeout. The client supplies its own timeouts and does not follow redirects with secret headers.
5. When finished, even if a check fails, run `modal app stop my-llm-service` and confirm in the dashboard that no containers remain. Revoke demo credentials when no longer needed.

`/docs` and `/openapi.json` are behind the same proxy gate. Use the header-capable client above rather than opening a bare URL. Reserve `Authorization` for the **app** token on `/chat`; never put proxy credentials in that header alongside a second bearer token.

**What local checks cannot prove:** a FastAPI `TestClient` calls the app directly and bypasses Modal's proxy. It can test app auth, the non-generating probe, validation, rate limits and token budgets; it cannot verify live edge admission, cold-start cost or GPU readiness. No deployment or live admission result is claimed here.


## Section 17.7: Secrets and Volumes

### Creating Secrets

Use the Modal dashboard to create `my-secrets` with an `API_KEY` field in the environment where its function will run. Reference the secret by name; do not hard-code or print the value. Proxy credentials belong on the trusted client, not in the function's secrets.

### Using Secrets in Code

In [ ]:
# Using secrets in your code
secrets_example = '''
@app.function(secrets=[modal.Secret.from_name("my-secrets")])
def with_secrets():
    import os
    api_key = os.environ["API_KEY"]
    # Use the key securely...
'''

print("Using secrets:")
print(secrets_example)

In [ ]:
# Using volumes to cache models
volumes_example = '''
# Create a volume for model cache
model_cache = modal.Volume.from_name("model-cache", create_if_missing=True)

@app.function(
    gpu="L4",
    volumes={"/root/.cache/huggingface": model_cache},
)
def with_cache():
    # Models download once, then cached in volume
    pass
'''

print("Using volumes to cache models:")
print(volumes_example)
print("\nFirst request downloads the model. Subsequent requests use the cache.")

## Section 17.8: GPU Options

| Model Size | Recommended GPU | Cost/Hour |
|------------|-----------------|----------|
| < 3B params (FP16/quantized) | T4 (16GB) | $0.59 |
| < 3B params (BF16, e.g. Qwen3.5-2B) | L4 (24GB) | $0.80 |
| 3-8B params | A10 (24GB) | $1.10 |
| 8-30B params | A100-40GB | $2.50 |
| 30B+ params | A100-80GB / H100 | $4-8 |

(Prices checked 2026-11; verify at modal.com/pricing.)

**Start with L4.** Qwen3.5-2B's weights are BF16, and vLLM 0.34 refuses BF16 on GPUs older than Ampere (compute capability 8.0): the T4 is 7.5 and would exit at startup, while L4 (8.9) and A10 (8.6) clear the bar. The T4 still works for smaller FP16 or quantized models. Upgrade only when needed.

In [ ]:
# GPU selection examples
print("GPU selection is one parameter:")
print()
print('@app.function(gpu="T4")      # 16GB, $0.59/hr - FP16 or quantized models only (no BF16)')
print('@app.function(gpu="L4")      # 24GB, $0.80/hr - our pick for Qwen3.5-2B')
print('@app.function(gpu="A10")     # 24GB, $1.10/hr - same memory, faster compute')
print('@app.function(gpu="A100")    # High performance')
print('@app.function(gpu="H100")    # Maximum power')
print('@app.function(gpu="A100:2")  # Two A100s')

## Section 17.9: Managing Your Deployment

### Viewing Logs

```bash
modal app logs my-llm-service
```

Or use the dashboard at [modal.com](https://modal.com).

### Updating Your App

```bash
# Just redeploy
modal deploy llm_service.py
```

No downtime. Modal handles rolling updates automatically.

### Cost Control

- Use `scaledown_window` to shut down idle containers
- Start with L4, upgrade only when needed
- Set spending alerts in the Modal dashboard; `modal app stop my-llm-service` tears everything down
- A request is limited to 4,000 characters, 2,048 templated input tokens and 512 output tokens. The locked 20-per-minute IP limiter is **per container** and resets on scale-down. These controls bound individual requests, not total spend across requests or replicas. For a shared service, use per-user quotas backed by atomic shared state; an alert is not a hard cap.

## Summary

You've learned how to:

1. **Deploy functions** to Modal with `@app.function()`
2. **Add GPU support** with a single parameter: `gpu="L4"`
3. **Install dependencies** using `modal.Image`
4. **Gate all web routes** before container startup with `@modal.asgi_app(requires_proxy_auth=True)`
5. **Store secrets** securely with `modal.Secret`
6. **Cache models** with `modal.Volume`
7. **Control costs** with idle timeouts and monitoring

**Share the interface, not your credentials.** A URL alone grants no access. Keep the workspace proxy pair on a trusted client or server, and the separate bearer token on callers authorized to generate. For end users, put your own authenticated gateway in front rather than distributing workspace credentials.

**Commands to remember:**
```bash
pip install modal      # Install
modal setup            # Authenticate
modal run file.py      # Run once
modal deploy file.py   # Deploy permanently
modal app logs name    # View logs
```